# NLinear - FX Pairs

NLinear forecasts from a fixed consecutive history after subtracting the last observed level. The
transformation focuses the model on changes over the lookback window. This notebook constructs
only the NLinear request; comparisons with TCN, TabM, trees, and linear models are deferred to
`12_model_analysis`, where the complete registered population is available.

**Learning objectives**

- Resolve NLinear's lookback, normalization, and checkpoint schedule before fitting.
- Use the shared gap-safe sequence eligibility instead of positional row windows.
- Prove weight reload and catalog handoff for every declared epoch.

**Book reference**: Chapter 13, Section 13.4

**Prerequisites**: `02_labels`, `03_financial_features`, and `04_model_based_features`.

In [1]:
"""Fit and catalog the published NLinear FX configuration."""

import json

import polars as pl
import yaml

from case_studies.research import ExecutionTier, Study, plan_models
from utils.modeling import load_configs
from utils.paths import get_case_study_dir
from utils.reproducibility import set_global_seeds

In [2]:
CASE_STUDY_ID = "fx_pairs"
PRIMARY_LABEL = ""
MAX_SYMBOLS = 0
MAX_FOLDS = 0
FORCE_RETRAIN = False
PREDICTION_SPLIT = "validation"
N_EPOCHS = 0
LOOKBACK = 0
BATCH_SIZE = 0
DEVICE = "cuda"
SEED = 42

## Resolve one forecasting request

The shared runner derives fold boundaries from the finalized label timeline. A missing daily
observation invalidates every lookback window that crosses it, so validation coverage can be
smaller than the raw validation panel while still being exact.

In [3]:
set_global_seeds(SEED)
case_dir = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((case_dir / "config" / "setup.yaml").read_text())
labels = (
    [PRIMARY_LABEL]
    if PRIMARY_LABEL
    else [setup["labels"]["primary"], *setup["labels"].get("variants", [])]
)

if PREDICTION_SPLIT != "validation":
    raise ValueError("model selection uses validation predictions; holdout runs start from a lock")
if FORCE_RETRAIN:
    raise ValueError("valid checkpoints are reloaded by identity; change the request to refit")

reductions = {
    **({"folds": list(range(MAX_FOLDS))} if MAX_FOLDS else {}),
    **({"max_symbols": MAX_SYMBOLS} if MAX_SYMBOLS else {}),
}
tier = ExecutionTier.PREVIEW if reductions else ExecutionTier.CANONICAL
overrides = {
    "device": DEVICE,
    **({"n_epochs": N_EPOCHS} if N_EPOCHS else {}),
    **({"batch_size": BATCH_SIZE} if BATCH_SIZE else {}),
    **({"lookback": LOOKBACK} if LOOKBACK else {}),
}
study = Study.regenerate(CASE_STUDY_ID)
ARCHITECTURE = "nlinear"
menu = {
    label: [
        config["config_name"]
        for config in load_configs(CASE_STUDY_ID, label, family="deep_learning")
    ]
    for label in labels
}
uncovered = {label: sorted(set(names) - {ARCHITECTURE}) for label, names in menu.items()}
for label, names in menu.items():
    if ARCHITECTURE not in names:
        raise RuntimeError(
            f"{ARCHITECTURE} is not in the configured deep_learning menu for {label}: {names}"
        )

requests = [
    study.model(
        family="deep_learning",
        label=label,
        config_name=ARCHITECTURE,
        execution_tier=tier,
        preview_reductions=reductions,
        overrides=overrides,
    )
    for label in labels
]
plan = plan_models(study, requests=requests)

# This notebook owes one architecture on every configured label. The rest of the family menu is
# named here rather than left implicit, because a population that is short a configured model is
# otherwise indistinguishable from a complete one.
configured = {(label, ARCHITECTURE) for label in labels}
planned = {(member.label, member.config_name) for member in plan.members}
if planned != configured:
    raise RuntimeError(
        f"the plan does not match this notebook's declared coverage; "
        f"missing {sorted(configured - planned)}, unexpected {sorted(planned - configured)}"
    )
specs = {member.label: json.loads(member.spec_json) for member in plan.members}
computations = {label: spec.get("computation", spec) for label, spec in specs.items()}
computation = computations[labels[0]]

print(f"Labels: {', '.join(labels)}")
print(f"Execution tier: {tier.value}")
print(f"Device: {computation['numerics']['device']}")
print(f"Lookback: {computation['preprocessing']['lookback']} consecutive daily observations")
for horizon, values in computations.items():
    print(f"Eligible validation rows, {horizon}: {values['expected_prediction_keys']['n_rows']:,}")
for horizon, names in uncovered.items():
    print(
        f"Configured deep_learning models this notebook does not run, {horizon}: {names or 'none'}"
    )

Labels: fwd_ret_1d, fwd_ret_5d, fwd_ret_21d
Execution tier: canonical
Device: cuda
Lookback: 60 consecutive daily observations
Eligible validation rows, fwd_ret_1d: 41,260
Eligible validation rows, fwd_ret_5d: 41,180
Eligible validation rows, fwd_ret_21d: 40,860
Configured deep_learning models this notebook does not run, fwd_ret_1d: ['lstm_h64', 'tcn']
Configured deep_learning models this notebook does not run, fwd_ret_5d: ['lstm_h64', 'tcn']
Configured deep_learning models this notebook does not run, fwd_ret_21d: ['lstm_h64', 'tcn']


## Inspect identity-bearing settings

The model request records its architecture parameters, exact folds, expected prediction-key
digest, and every epoch that must remain reproducible from stored weights.

In [4]:
checkpoint_schedule = pl.DataFrame(computation["checkpoint_schedule"])
pl.DataFrame(
    {
        "label": list(computations),
        "architecture": [c["model"]["class"] for c in computations.values()],
        "gap_policy": [c["preprocessing"]["gap_policy"] for c in computations.values()],
        "validation_folds": [
            c["expected_prediction_keys"]["n_folds"] for c in computations.values()
        ],
        "key_digest": [c["expected_prediction_keys"]["digest"] for c in computations.values()],
    }
)
checkpoint_schedule

kind,value
str,i64
"""epoch""",5
"""epoch""",10
"""epoch""",15
"""epoch""",20
"""epoch""",25
…,…
"""epoch""",80
"""epoch""",85
"""epoch""",90


## Record the official population, then fit or reload NLinear

The runner validates every fold separately before any checkpoint becomes downstream-selectable.
Checkpoint rank correlation is retained as a diagnostic and does not remove other epochs.

In [5]:
if len(plan.expected_prediction_hashes) != checkpoint_schedule.height * len(labels):
    raise RuntimeError("the plan does not cover every declared epoch checkpoint on every label")
population = (
    plan.create_population(name=f"{CASE_STUDY_ID}:{'+'.join(labels)}:nlinear")
    if tier is ExecutionTier.CANONICAL
    else None
)

execution = plan.run()
catalog = execution.catalog_rows.sort("label", "checkpoint_value")
if set(catalog.get_column("prediction_hash")) != set(plan.expected_prediction_hashes):
    raise RuntimeError("the published catalog differs from the population planned before fitting")
if catalog.filter(~pl.col("complete")).height:
    raise RuntimeError("partial NLinear checkpoints cannot pass to backtesting")
for label in labels:
    published = catalog.filter(pl.col("label") == label).get_column("checkpoint_value").to_list()
    if published != checkpoint_schedule["value"].to_list():
        raise RuntimeError(f"catalog checkpoints for {label} differ from the resolved request")

catalog.select(
    "label",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "complete",
    "ic_mean",
    "ic_t",
    "training_hash",
    "prediction_hash",
)

Fold-major CV: 8 folds × 1 configs × 60 lookback

  Fold 0: creating sequences...


    train=24,580 seq across 20 symbols
    val=5,140 seq across 20 symbols
    creating datasets...
    datasets ready
    nlinear:


      epoch   1/100: train_loss=0.499876


      epoch   2/100: train_loss=0.240427


      epoch   3/100: train_loss=0.109742


...

      epoch 100/100: train_loss=0.000798, val_loss=0.001228, IC=+0.0145


      best_ep=55, IC=+0.0291 (28.2s, 20 checkpoints)


  nlinear: best_epoch=5, IC=+0.0344 (330.5s)



  Best: nlinear @ epoch 5 (IC=+0.0344)
  Saved to ~/ml4t/public-s6-fx_pairs/case_studies/fx_pairs/run_log/training/1d65a7393f44/diagnostics


label,config_name,checkpoint_kind,checkpoint_value,complete,ic_mean,ic_t,training_hash,prediction_hash
str,str,str,i64,bool,f64,f64,str,str
"""fwd_ret_1d""","""nlinear""","""epoch""",5,true,0.013303,1.723168,"""e9348cbe2199""","""9eb5dd8577d1"""
"""fwd_ret_1d""","""nlinear""","""epoch""",10,true,0.014516,2.158379,"""e9348cbe2199""","""26150ab0d3db"""
"""fwd_ret_1d""","""nlinear""","""epoch""",15,true,0.013679,2.06296,"""e9348cbe2199""","""70cc489e40cd"""
"""fwd_ret_1d""","""nlinear""","""epoch""",20,true,0.01171,1.781676,"""e9348cbe2199""","""9eae73b86751"""
"""fwd_ret_1d""","""nlinear""","""epoch""",25,true,0.012545,1.851869,"""e9348cbe2199""","""1df5922d71dc"""
…,…,…,…,…,…,…,…,…
"""fwd_ret_5d""","""nlinear""","""epoch""",80,true,0.013017,0.757352,"""c845bed56399""","""0c5ecb57cb04"""
"""fwd_ret_5d""","""nlinear""","""epoch""",85,true,0.010603,0.639332,"""c845bed56399""","""099830b9220a"""
"""fwd_ret_5d""","""nlinear""","""epoch""",90,true,0.011746,0.720837,"""c845bed56399""","""cba7f1e37e22"""


## Verify checkpoint reload

Repeating the request validates the fitted-state digests and returns the same prediction
identities. The notebook never reconstructs another family from an empty cache path.

In [6]:
replayed = plan.run()
if set(replayed.catalog_rows.get_column("prediction_hash")) != set(
    catalog.get_column("prediction_hash")
):
    raise RuntimeError("NLinear checkpoint reload changed the prediction population")

if population is not None:
    population.require_complete()
    print(f"Official prediction population: {population.hash}")
else:
    print("Preview sequence checkpoints remain outside official comparisons.")

Official prediction population: 7cf7405afc08


## Key takeaways

- NLinear and TCN use the same sequence eligibility contract but keep separate model identities.
- Gaps remove affected windows instead of being hidden by positional indexing.
- Stored weights reproduce every declared checkpoint without retraining.